[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/17_rmsnorm_solution.ipynb)

# 🟢 Solution: RMSNorm (nnx.Module)

*Core Ops & Layers · Easy*

Reference implementation. Try it yourself in `17_rmsnorm.ipynb` first.

---
Implement **RMSNorm** as an `nnx.Module`.

$$y = \frac{x}{\sqrt{\frac{1}{d}\sum_i x_i^2 + \epsilon}} \cdot \gamma$$

### Rules
- Subclass `nnx.Module`
- Signature: `RMSNorm(dim, *, eps=1e-6, rngs=None)`
- One parameter only: `self.scale`, initialised to ones
- **No mean subtraction** and **no bias** — that is the whole point
- Normalise over the last axis

### Why drop the mean
RMSNorm is LayerNorm with the re-centering removed. The 2019 paper's finding was
that the *re-scaling* is what stabilises training; the *re-centering* contributes
almost nothing. Dropping it saves a pass over the data and a subtraction, which
at Llama scale is a real win.

Llama, Mistral, Gemma, and T5 all use RMSNorm. GPT-2 and BERT use LayerNorm.

The follow-up worth being ready for: **when do the two coincide?** Exactly when
the input already has zero mean — then $\text{RMS}(x) = \sigma(x)$ and the two
are identical up to the missing bias.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from flax import nnx


class RMSNorm(nnx.Module):
    def __init__(self, dim: int, *, eps: float = 1e-6, rngs: nnx.Rngs = None):
        self.scale = nnx.Param(jnp.ones((dim,)))   # no bias term
        self.eps = eps
        self.dim = dim

    def __call__(self, x):
        # Mean of the squares — no mean subtraction anywhere.
        ms = jnp.mean(jnp.square(x), axis=-1, keepdims=True)
        return x / jnp.sqrt(ms + self.eps) * self.scale

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp
from flax import nnx

rms = RMSNorm(8)
x = jax.random.normal(jax.random.key(0), (2, 8)) * 3.0 + 10.0

out = rms(x)
print("input  RMS per row:", jnp.sqrt(jnp.mean(x ** 2, -1)))
print("output RMS per row:", jnp.sqrt(jnp.mean(out ** 2, -1)), "(~1)")
print("output mean per row:", out.mean(-1), "(NOT ~0 — no re-centering)")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("rmsnorm")